In [ ]:
!pip install -q chatterbox-tts torchvision==0.21.0 peft indic-transliteration indic-num2words

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"          # keep ~3GB of weights out of your 20GB output quota
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch, torchaudio, numpy
print("torch", torch.__version__, "| torchaudio", torchaudio.__version__, "| numpy", numpy.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [3]:
!pip uninstall -y torchao
!rm -rf /usr/local/lib/python3.12/dist-packages/torchao /usr/local/lib/python3.12/dist-packages/torchao-*.dist-info

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip uninstall -y numpy
!rm -rf /usr/local/lib/python3.12/dist-packages/numpy /usr/local/lib/python3.12/dist-packages/numpy-*.dist-info /usr/local/lib/python3.12/dist-packages/numpy.libs
!pip install -q --no-cache-dir "numpy==1.26.4"

In [4]:
import re, time, torch
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

REPO, DEVICE = "Praxel/praxy-voice-r6", "cuda"

def load_model(with_lora=True):
    model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
    if not with_lora:
        return model

    for p in model.t3.parameters():
        p.requires_grad_(False)
    model.t3 = get_peft_model(model.t3, LoraConfig(
        r=32, lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
    ))

    sd = torch.load(hf_hub_download(REPO, "lora_state.pt"), map_location=DEVICE)
    sd = sd.get("state_dict", sd)

    # strict=False silently loads NOTHING on a key mismatch -> you'd hear vanilla
    # Chatterbox and call it a success. Check before trusting any output.
    tgt = set(model.t3.state_dict().keys())
    in_file = [k for k in sd if "lora_" in k]
    matched = [k for k in in_file if k in tgt]
    print(f"[lora] {len(in_file)} lora tensors in file, {len(matched)} matched")
    if in_file and not matched:
        print(" file key :", in_file[0])
        print(" model key:", next(k for k in tgt if "lora_" in k))
        raise RuntimeError("Key mismatch - adapter would be a no-op.")

    model.t3.load_state_dict(sd, strict=False, assign=True)
    model.t3.eval()
    # lora_B is zero-init; nonzero sum proves trained weights are in place.
    s = sum(float(model.t3.state_dict()[k].abs().sum()) for k in matched if "lora_B" in k)
    print(f"[lora] sum|lora_B| = {s:.4f}  (must be > 0)")
    return model

model = load_model(with_lora=True)
print("sample rate:", model.sr)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

loaded PerthNet (Implicit) at step 250,000


lora_state.pt:   0%|          | 0.00/31.5M [00:00<?, ?B/s]

[lora] 240 lora tensors in file, 240 matched
[lora] sum|lora_B| = 7074.7440  (must be > 0)
sample rate: 24000


In [5]:
import glob, torchaudio

cands = glob.glob("/kaggle/input/**/*.wav", recursive=True) + \
        glob.glob("/kaggle/input/**/*.mp3", recursive=True)
print("found:", cands)

SRC = cands[0]                     # <-- point at your Telugu/Tamil clip
REF = "/kaggle/working/ref_9s.wav"

wav, sr = torchaudio.load(SRC)
wav = wav.mean(0, keepdim=True)                                  # mono
if sr != 24000:
    wav, sr = torchaudio.functional.resample(wav, sr, 24000), 24000
wav = wav[:, : int(9 * sr)]                                      # 9s, mid-range of 8-11
torchaudio.save(REF, wav, sr)
print(f"ref: {wav.shape[-1]/sr:.1f}s @ {sr}Hz -> {REF}")

from IPython.display import Audio, display
display(Audio(REF))

found: ['/kaggle/input/datasets/henil2132/telgu-audio/telgu.wav']
ref: 9.0s @ 24000Hz -> /kaggle/working/ref_9s.wav


In [7]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from num_to_words import num_to_word
from IPython.display import Audio, display

SCRIPTS  = {"te": sanscript.TELUGU, "ta": sanscript.TAMIL, "hi": sanscript.DEVANAGARI}
SAMPLING = dict(exaggeration=0.7, temperature=0.6, min_p=0.1,
                cfg_weight=0.5, repetition_penalty=2.0, top_p=1.0)

LANG = "te"
TEXT = "నేను ఇవాళ బాగున్నాను. మీ అపాయింట్‌మెంట్ 3 గంటలకు ఉంది."

norm = re.sub(r"\d+", lambda m: num_to_word(int(m.group(0)), LANG).replace(",", "").strip(), TEXT)
payload = transliterate(norm, SCRIPTS[LANG], sanscript.ISO)     # BUPS / ISO-15919
print("sent to model:", payload)

def render(m, tag):
    t0 = time.time()
    with torch.inference_mode():
        # language_id is "hi" even for te/ta - that's the card's proxy routing, not a typo
        w = m.generate(payload, language_id="hi", audio_prompt_path=REF, **SAMPLING)
    w = w.detach().cpu()
    if w.ndim == 1: w = w.unsqueeze(0)
    out = f"/kaggle/working/out_{LANG}_{tag}.wav"
    torchaudio.save(out, w, m.sr)
    dur = w.shape[-1] / m.sr
    print(f"[{tag}] {dur:.2f}s audio in {time.time()-t0:.1f}s (RTF {(time.time()-t0)/dur:.2f})")
    display(Audio(out))
    return out

render(model, "lora")

sent to model: nēnu ivāḷa bāgunnānu. mī apāyiṁṭ‌meṁṭ mūḍu gaṁṭalaku uṁdi.


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Sampling:  14%|█▍        | 140/1000 [00:06<00:38, 22.23it/s]


[lora] 5.60s audio in 18.8s (RTF 3.36)


'/kaggle/working/out_te_lora.wav'

In [2]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from num_to_words import num_to_word
from IPython.display import Audio, display

SCRIPTS  = {"te": sanscript.TELUGU, "ta": sanscript.TAMIL, "hi": sanscript.DEVANAGARI}
SAMPLING = dict(exaggeration=0.7, temperature=0.6, min_p=0.1,
                cfg_weight=0.5, repetition_penalty=2.0, top_p=1.0)

LANG = "te"
TEXT = "ఈ వైద్య పుస్తకం మానవ శరీరం, వైద్యులు, వ్యాధులు, కారణాలు"

norm = re.sub(r"\d+", lambda m: num_to_word(int(m.group(0)), LANG).replace(",", "").strip(), TEXT)
payload = transliterate(norm, SCRIPTS[LANG], sanscript.ISO)     # BUPS / ISO-15919
print("sent to model:", payload)

def render(m, tag):
    t0 = time.time()
    with torch.inference_mode():
        # language_id is "hi" even for te/ta - that's the card's proxy routing, not a typo
        w = m.generate(payload, language_id="hi", audio_prompt_path=REF, **SAMPLING)
    w = w.detach().cpu()
    if w.ndim == 1: w = w.unsqueeze(0)
    out = f"/kaggle/working/out_{LANG}_{tag}.wav"
    torchaudio.save(out, w, m.sr)
    dur = w.shape[-1] / m.sr
    print(f"[{tag}] {dur:.2f}s audio in {time.time()-t0:.1f}s (RTF {(time.time()-t0)/dur:.2f})")
    display(Audio(out))
    return out

render(model, "lora")

ModuleNotFoundError: No module named 'indic_transliteration'

In [8]:
del model; torch.cuda.empty_cache()
render(load_model(with_lora=False), "vanilla")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000


Sampling:  22%|██▏       | 216/1000 [00:06<00:24, 32.26it/s]


[vanilla] 8.64s audio in 8.8s (RTF 1.01)


'/kaggle/working/out_te_vanilla.wav'